In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

from matplotlib.colors import LogNorm

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
EXTERNAL_DATA_DIR = DATA_DIR / "external"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"
REPORTS_DIR = OUTPUT_DIR / "reports"

DATA_PATH = RAW_DATA_DIR / "yellow_tripdata_2026-03.parquet"

ZONE_MAP_PATH = EXTERNAL_DATA_DIR / "taxi_zones.shp"
ZONE_LOOKUP_PATH = EXTERNAL_DATA_DIR / "taxi_zone_lookup.csv"

for directory in [
    PROCESSED_DATA_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    REPORTS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())
print("Zone shapefile exists:", ZONE_MAP_PATH.exists())
print("Zone lookup exists:", ZONE_LOOKUP_PATH.exists())

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

df = pd.read_parquet(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {len(df):,}")
print(f"Number of columns: {df.shape[1]}")

display(df.head())

In [ ]:
df.info()

In [ ]:
dataset_structure = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_percent": (
        df.isna().mean().mul(100).round(2).values
    ),
    "unique_values": df.nunique().values
})

display(dataset_structure)

In [ ]:
dataset_structure.to_csv(
    TABLES_DIR / "raw_dataset_structure.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Initial dataset structure saved.")

In [ ]:
START_DATE = pd.Timestamp("2026-03-01")
END_DATE = pd.Timestamp("2026-04-01")

march_mask = (
    df["tpep_pickup_datetime"].ge(START_DATE)
    & df["tpep_pickup_datetime"].lt(END_DATE)
)

df_mar = df.loc[march_mask].copy()

print(f"Original rows: {len(df):,}")
print(f"March rows: {len(df_mar):,}")
print(f"Excluded rows: {len(df) - len(df_mar):,}")

In [ ]:
print("Pickup datetime range:")
print(df_mar["tpep_pickup_datetime"].min())
print(df_mar["tpep_pickup_datetime"].max())

print("\nDropoff datetime range:")
print(df_mar["tpep_dropoff_datetime"].min())
print(df_mar["tpep_dropoff_datetime"].max())

print("\nNumber of records:", f"{len(df_mar):,}")
print("Number of columns:", df_mar.shape[1])

In [ ]:
outside_march = (
    df_mar["tpep_pickup_datetime"].lt(START_DATE)
    | df_mar["tpep_pickup_datetime"].ge(END_DATE)
)

print("Records outside March:", int(outside_march.sum()))

assert len(df_mar) > 0, "Filtered dataset is empty."
assert outside_march.sum() == 0, "Out-of-range records detected."

print("Date filter validation passed.")

In [ ]:
dataset_overview = pd.DataFrame({
    "item": [
        "dataset_name",
        "source",
        "source_file",
        "analysis_period",
        "filtering_variable",
        "number_of_records",
        "number_of_columns",
        "exact_duplicate_rows"
    ],
    "value": [
        "NYC TLC Yellow Taxi Trip Records",
        "NYC Taxi and Limousine Commission",
        DATA_PATH.name,
        "2026-03-01 to 2026-03-31",
        "tpep_pickup_datetime",
        len(df_mar),
        df_mar.shape[1],
        int(df_mar.duplicated().sum())
    ]
})

display(dataset_overview)

In [ ]:
dataset_overview.to_csv(
    TABLES_DIR / "dataset_overview_march_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset overview saved.")

In [ ]:
data_dictionary = pd.DataFrame({
    "column": df_mar.columns,
    "data_type": df_mar.dtypes.astype(str).values,
    "missing_count": df_mar.isna().sum().values,
    "missing_percent": (
        df_mar.isna().mean().mul(100).round(2).values
    ),
    "unique_values": df_mar.nunique().values
})

display(data_dictionary)

In [ ]:
data_dictionary.to_csv(
    TABLES_DIR / "data_dictionary_march_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Data dictionary saved.")

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df_mar.isna().sum(),
    "missing_percent": (
        df_mar.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_percent",
    ascending=False
)

display(missing_summary)

In [ ]:
duplicate_count = df_mar.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_count:,}")
print(
    "Duplicate percentage:",
    round(duplicate_count / len(df_mar) * 100, 4),
    "%"
)

In [ ]:
missing_cols = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

missing_pattern = df_mar[missing_cols].isna()

missing_per_row = missing_pattern.sum(axis=1)

missing_pattern_summary = (
    missing_per_row
    .value_counts()
    .sort_index()
    .rename_axis("missing_columns_per_row")
    .to_frame("row_count")
)

display(missing_pattern_summary)

all_five_missing = missing_pattern.all(axis=1)

print(
    "Rows with all five columns missing:",
    f"{all_five_missing.sum():,}"
)

In [ ]:
print("Payment type among rows with all five values missing:")

display(
    df_mar.loc[all_five_missing, "payment_type"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("payment_type")
    .to_frame("row_count")
)

In [ ]:
df_mar["trip_duration_min"] = (
    df_mar["tpep_dropoff_datetime"]
    - df_mar["tpep_pickup_datetime"]
).dt.total_seconds() / 60

print("Trip duration calculated.")
display(df_mar["trip_duration_min"].describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
).to_frame("trip_duration_min"))

In [ ]:
duration_checks = {
    "missing_pickup_time":
        df_mar["tpep_pickup_datetime"].isna().sum(),

    "missing_dropoff_time":
        df_mar["tpep_dropoff_datetime"].isna().sum(),

    "negative_duration":
        (df_mar["trip_duration_min"] < 0).sum(),

    "zero_duration":
        (df_mar["trip_duration_min"] == 0).sum(),

    "duration_over_4_hours":
        (df_mar["trip_duration_min"] > 240).sum()
}

display(
    pd.Series(
        duration_checks,
        name="record_count"
    ).to_frame()
)

In [ ]:
negative_duration = df_mar.loc[
    df_mar["trip_duration_min"] < 0,
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "payment_type"
    ]
]

display(negative_duration)

In [ ]:
long_duration = (
    df_mar.loc[
        df_mar["trip_duration_min"] > 240,
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_distance",
            "fare_amount",
            "total_amount",
            "payment_type",
            "trip_duration_min"
        ]
    ]
    .sort_values("trip_duration_min", ascending=False)
)

display(long_duration.head(15))

In [ ]:
quality_checks = {
    "negative_distance":
        (df_mar["trip_distance"] < 0).sum(),

    "zero_distance":
        (df_mar["trip_distance"] == 0).sum(),

    "negative_fare":
        (df_mar["fare_amount"] < 0).sum(),

    "zero_fare":
        (df_mar["fare_amount"] == 0).sum(),

    "negative_tip":
        (df_mar["tip_amount"] < 0).sum(),

    "negative_tolls":
        (df_mar["tolls_amount"] < 0).sum(),

    "negative_total_amount":
        (df_mar["total_amount"] < 0).sum(),

    "zero_total_amount":
        (df_mar["total_amount"] == 0).sum()
}

quality_summary = pd.DataFrame({
    "check": list(quality_checks.keys()),
    "record_count": list(quality_checks.values())
})

quality_summary["percent_of_trips"] = (
    quality_summary["record_count"] / len(df_mar) * 100
)

quality_summary["percent_of_trips"] = (
    quality_summary["percent_of_trips"].round(3)
)

display(quality_summary)

In [ ]:
negative_amounts = df_mar.loc[
    (df_mar["fare_amount"] < 0)
    | (df_mar["total_amount"] < 0)
].copy()

payment_quality = (
    negative_amounts
    .groupby("payment_type")
    .agg(
        record_count=("payment_type", "size"),
        negative_fare_count=(
            "fare_amount",
            lambda x: (x < 0).sum()
        ),
        negative_total_count=(
            "total_amount",
            lambda x: (x < 0).sum()
        )
    )
    .reset_index()
)

payment_quality["share_percent"] = (
    payment_quality["record_count"]
    / max(len(negative_amounts), 1)
    * 100
).round(2)

display(payment_quality)

In [ ]:
missing_summary.to_csv(
    TABLES_DIR / "missing_values_march_2026.csv",
    encoding="utf-8-sig"
)

quality_summary.to_csv(
    TABLES_DIR / "data_quality_summary_march_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

payment_quality.to_csv(
    TABLES_DIR / "negative_amounts_by_payment_type.csv",
    index=False,
    encoding="utf-8-sig"
)

cleaning_rules = pd.DataFrame([
    {
        "issue": "Missing timestamps",
        "decision": "Report and investigate",
        "reason": "Timestamps are required for temporal analysis"
    },
    {
        "issue": "Exact duplicate rows",
        "decision": "Do not remove automatically",
        "reason": "No duplicates were observed in the previous analysis"
    },
    {
        "issue": "Negative trip duration",
        "decision": "Flag for investigation",
        "reason": "May indicate inconsistent timestamps"
    },
    {
        "issue": "Zero duration or distance",
        "decision": "Retain and investigate",
        "reason": "Not necessarily invalid without further evidence"
    },
    {
        "issue": "Negative fare or total amount",
        "decision": "Retain and investigate",
        "reason": "Could represent adjustments or other payment-related cases"
    },
    {
        "issue": "Missing values in five columns",
        "decision": "Do not impute automatically",
        "reason": "Investigate the shared missingness pattern first"
    }
])

cleaning_rules.to_csv(
    TABLES_DIR / "cleaning_decisions_march_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Quality reports saved successfully.")

In [ ]:
numeric_cols = [
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "tolls_amount",
    "passenger_count",
    "trip_duration_min"
]

numeric_cols = [
    col for col in numeric_cols
    if col in df_mar.columns
]

print("Variables for descriptive analysis:")
print(numeric_cols)

In [ ]:
descriptive_stats = (
    df_mar[numeric_cols]
    .describe(
        percentiles=[
            0.01, 0.25, 0.50, 0.75, 0.95, 0.99
        ]
    )
    .T
)

descriptive_stats["missing_count"] = (
    df_mar[numeric_cols].isna().sum()
)

descriptive_stats["zero_count"] = (
    df_mar[numeric_cols].eq(0).sum()
)

descriptive_stats["negative_count"] = (
    df_mar[numeric_cols].lt(0).sum()
)

display(descriptive_stats.round(2))

In [ ]:
descriptive_stats.to_csv(
    TABLES_DIR / "numeric_descriptive_statistics_march_2026.csv",
    encoding="utf-8-sig"
)

print("Descriptive statistics saved.")

In [ ]:
plot_cols = [
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "trip_duration_min"
]

fig, axes = plt.subplots(
    nrows=len(plot_cols),
    ncols=1,
    figsize=(12, 18)
)

for ax, col in zip(axes, plot_cols):
    series = df_mar[col].dropna()

    upper_limit = series.quantile(0.99)

    plot_data = series[
        (series >= 0) &
        (series <= upper_limit)
    ]

    sns.histplot(
        data=plot_data,
        bins=60,
        ax=ax
    )

    ax.set_title(
        f"Distribution of {col} (0 to 99th percentile)"
    )
    ax.set_xlabel(col)
    ax.set_ylabel("Number of trips")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "numeric_distributions_march_2026.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
passenger_counts = (
    df_mar["passenger_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("passenger_count")
    .to_frame("trip_count")
)

passenger_counts["share_percent"] = (
    passenger_counts["trip_count"]
    / len(df_mar)
    * 100
).round(2)

display(passenger_counts)

In [ ]:
passenger_plot_data = (
    df_mar["passenger_count"]
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(10, 5))

passenger_plot_data.plot(
    kind="bar",
    ax=ax
)

ax.set_title("Passenger Count Distribution — March 2026")
ax.set_xlabel("Passenger Count")
ax.set_ylabel("Number of Trips")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "passenger_count_distribution_march_2026.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
eda_cols = [
    "trip_distance",
    "fare_amount",
    "trip_duration_min"
]

upper_limits = df_mar[eda_cols].quantile(0.99)

valid_mask = (
    df_mar[eda_cols].notna().all(axis=1)
    & (df_mar[eda_cols] > 0).all(axis=1)
    & (df_mar[eda_cols] <= upper_limits).all(axis=1)
)

eda_data = df_mar.loc[valid_mask, eda_cols].copy()

print("Original March records:", f"{len(df_mar):,}")
print("Records selected for multivariate analysis:", f"{len(eda_data):,}")
print("Excluded from this analysis:", f"{len(df_mar) - len(eda_data):,}")

display(eda_data.describe().round(2))

In [ ]:
eda_sample = eda_data.sample(
    n=min(30_000, len(eda_data)),
    random_state=42
)

print("Scatter plot sample size:", f"{len(eda_sample):,}")

In [ ]:
pair_plot = sns.pairplot(
    eda_sample,
    vars=[
        "trip_distance",
        "fare_amount",
        "trip_duration_min"
    ],
    corner=True,
    plot_kws={
        "alpha": 0.25,
        "s": 10
    }
)

pair_plot.fig.suptitle(
    "Relationships Between Trip Distance, Fare and Duration",
    y=1.02
)

pair_plot.fig.savefig(
    FIGURES_DIR / "multivariate_pairplot_march_2026.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
pearson_corr = eda_data.corr(
    method="pearson"
)

display(pearson_corr.round(3))

pearson_corr.to_csv(
    TABLES_DIR / "pearson_correlation_march_2026.csv",
    encoding="utf-8-sig"
)

In [ ]:
spearman_corr = eda_data.corr(
    method="spearman"
)

display(spearman_corr.round(3))

spearman_corr.to_csv(
    TABLES_DIR / "spearman_correlation_march_2026.csv",
    encoding="utf-8-sig"
)

In [ ]:
fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(14, 5)
)

sns.heatmap(
    pearson_corr,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    ax=axes[0]
)

axes[0].set_title("Pearson Correlation")

sns.heatmap(
    spearman_corr,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    ax=axes[1]
)

axes[1].set_title("Spearman Correlation")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "correlation_heatmaps_march_2026.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
pairs = [
    ("trip_distance", "fare_amount"),
    ("trip_distance", "trip_duration_min"),
    ("fare_amount", "trip_duration_min")
]

correlation_comparison = pd.DataFrame([
    {
        "variable_1": var1,
        "variable_2": var2,
        "pearson": pearson_corr.loc[var1, var2],
        "spearman": spearman_corr.loc[var1, var2]
    }
    for var1, var2 in pairs
])

display(correlation_comparison.round(3))

correlation_comparison.to_csv(
    TABLES_DIR / "correlation_comparison_march_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
daily_demand = (
    df_mar
    .set_index("tpep_pickup_datetime")
    .resample("D")
    .size()
    .rename("trip_count")
    .asfreq("D")
)

print("Number of days:", len(daily_demand))
print("Missing days:", daily_demand.isna().sum())
print(
    "Date range:",
    daily_demand.index.min(),
    "to",
    daily_demand.index.max()
)

display(daily_demand.describe().to_frame())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

daily_demand.plot(
    ax=ax,
    marker="o",
    linewidth=1.5
)

ax.set_title("NYC Yellow Taxi Daily Demand — March 2026")
ax.set_xlabel("Date")
ax.set_ylabel("Number of Trips")
ax.grid(alpha=0.3)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "daily_demand_march_2026.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print("Top 5 highest-demand days:")
display(
    daily_demand.nlargest(5)
    .rename_axis("date")
    .to_frame()
)

print("Top 5 lowest-demand days:")
display(
    daily_demand.nsmallest(5)
    .rename_axis("date")
    .to_frame()
)

In [ ]:
weekday_names = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_summary = (
    daily_demand.to_frame()
    .assign(day_of_week=lambda x: x.index.dayofweek)
    .groupby("day_of_week")["trip_count"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reindex(range(7))
)

weekday_summary.index = weekday_names

display(weekday_summary.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

weekday_summary["mean"].plot(
    kind="bar",
    ax=ax
)

ax.set_title("Average Daily Taxi Demand by Weekday")
ax.set_xlabel("Day of Week")
ax.set_ylabel("Average Number of Trips")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "weekday_demand_march_2026.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
hourly_demand = (
    df_mar
    .groupby(df_mar["tpep_pickup_datetime"].dt.hour)
    .size()
    .reindex(range(24), fill_value=0)
    .rename("trip_count")
    .to_frame()
)

hourly_demand.index.name = "pickup_hour"

hourly_demand["share_percent"] = (
    hourly_demand["trip_count"]
    / len(df_mar)
    * 100
)

display(
    hourly_demand
    .sort_values("trip_count", ascending=False)
    .head(10)
    .round(2)
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(
    hourly_demand.index,
    hourly_demand["trip_count"]
)

ax.set_title("NYC Yellow Taxi Demand by Pickup Hour — March 2026")
ax.set_xlabel("Pickup Hour (0–23)")
ax.set_ylabel("Number of Trips")
ax.set_xticks(range(24))
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "hourly_demand_march_2026.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
df_mar["day_of_week"] = (
    df_mar["tpep_pickup_datetime"].dt.dayofweek
)

df_mar["pickup_hour"] = (
    df_mar["tpep_pickup_datetime"].dt.hour
)

hourly_weekday_demand = (
    df_mar
    .pivot_table(
        index="day_of_week",
        columns="pickup_hour",
        aggfunc="size",
        fill_value=0
    )
    .reindex(
        index=range(7),
        columns=range(24),
        fill_value=0
    )
)

hourly_weekday_demand.index = weekday_names

display(hourly_weekday_demand)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))

sns.heatmap(
    hourly_weekday_demand,
    cmap="YlOrRd",
    linewidths=0.3,
    annot=True,
    fmt=".0f",
    linecolor="white",
    cbar_kws={"label": "Number of Trips"},
    ax=ax
)

ax.set_xlabel("Pickup Hour")
ax.set_ylabel("Day of Week")
ax.set_title("NYC Yellow Taxi Demand by Weekday and Hour")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "weekday_hour_demand_march_2026.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
daily_demand.rename_axis("date").to_csv(
    TABLES_DIR / "daily_demand_march_2026.csv",
    encoding="utf-8-sig"
)

weekday_summary.to_csv(
    TABLES_DIR / "weekday_demand_march_2026.csv",
    encoding="utf-8-sig"
)

hourly_demand.to_csv(
    TABLES_DIR / "hourly_demand_march_2026.csv",
    encoding="utf-8-sig"
)

hourly_weekday_demand.to_csv(
    TABLES_DIR / "weekday_hour_demand_march_2026.csv",
    encoding="utf-8-sig"
)

print("Temporal analysis tables saved successfully.")

In [ ]:
zones_geo = gpd.read_file(ZONE_MAP_PATH)
zones_lookup = pd.read_csv(ZONE_LOOKUP_PATH)

print("Geographic data shape:", zones_geo.shape)
print("Zone lookup shape:", zones_lookup.shape)

print("\nGeographic columns:")
print(zones_geo.columns.tolist())

print("\nLookup columns:")
print(zones_lookup.columns.tolist())

print("\nCoordinate reference system:")
print(zones_geo.crs)

display(zones_geo.head())
display(zones_lookup.head())

In [ ]:

print("Missing LocationID in shapefile:",
      zones_geo["LocationID"].isna().sum())

print("Duplicate LocationID in shapefile:",
      zones_geo["LocationID"].duplicated().sum())

print("Missing LocationID in lookup:",
      zones_lookup["LocationID"].isna().sum())

print("Duplicate LocationID in lookup:",
      zones_lookup["LocationID"].duplicated().sum())

print("\nMissing geometries:",
      zones_geo.geometry.isna().sum())

print("Empty geometries:",
      zones_geo.geometry.is_empty.sum())

print("\nNumber of unique geographic zones:",
      zones_geo["LocationID"].nunique())

print("Number of unique lookup zones:",
      zones_lookup["LocationID"].nunique())

In [ ]:
pickup_counts = (
    df_mar
    .groupby("PULocationID")
    .size()
    .rename("trip_count")
    .reset_index()
    .rename(columns={"PULocationID": "LocationID"})
)

pickup_counts["LocationID"] = pickup_counts["LocationID"].astype("int64")

print("Number of zones with pickup activity:",
      pickup_counts["LocationID"].nunique())

print("Total trips counted:", pickup_counts["trip_count"].sum())

display(pickup_counts.head())

In [ ]:
zones_geo["LocationID"] = zones_geo["LocationID"].astype("int64")
zones_lookup["LocationID"] = zones_lookup["LocationID"].astype("int64")

zones_demand = zones_geo.merge(
    pickup_counts,
    on="LocationID",
    how="left",
    validate="one_to_one"
)

zones_demand["trip_count"] = (
    zones_demand["trip_count"]
    .fillna(0)
    .astype("int64")
)

print("Merged geographic data shape:", zones_demand.shape)
print("Zones with zero recorded pickups:",
      (zones_demand["trip_count"] == 0).sum())

display(
    zones_demand[
        ["LocationID", "zone", "borough", "trip_count"]
    ].head()
)

In [ ]:
top_pickup_zones = (
    pickup_counts
    .merge(
        zones_lookup[["LocationID", "Zone", "Borough"]],
        on="LocationID",
        how="left",
        validate="one_to_one"
    )
    .sort_values("trip_count", ascending=False)
    .reset_index(drop=True)
)

top_pickup_zones["share_percent"] = (
    top_pickup_zones["trip_count"]
    / len(df_mar)
    * 100
)

top_pickup_zones.to_csv(
    TABLES_DIR / "top_pickup_zones_march_2026.csv",
    index=False
)

display(top_pickup_zones.head(15))

In [ ]:
zones_plot = zones_demand.copy()

zones_plot["trip_count_log"] = zones_plot["trip_count"].where(
    zones_plot["trip_count"] > 0
)

fig, ax = plt.subplots(figsize=(14, 12))

zones_plot.plot(
    column="trip_count_log",
    cmap="viridis",
    linewidth=0.15,
    edgecolor="white",
    legend=True,
    norm=LogNorm(
        vmin=1,
        vmax=max(1, zones_plot["trip_count"].max())
    ),
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "white",
        "label": "No recorded pickups"
    },
    legend_kwds={
        "label": "Number of pickup trips (log scale)",
        "shrink": 0.65
    },
    ax=ax
)

ax.set_title(
    "NYC Yellow Taxi Pickup Demand by Zone — March 2026",
    fontsize=15
)
ax.set_axis_off()

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "pickup_demand_map_log_march_2026.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
zone_demand_summary = pd.DataFrame({
    "metric": [
        "Number of zones with pickup activity",
        "Total pickup trips",
        "Mean trips per active zone",
        "Median trips per active zone",
        "Maximum trips in one zone",
        "Minimum trips in active zones"
    ],
    "value": [
        pickup_counts["LocationID"].nunique(),
        pickup_counts["trip_count"].sum(),
        pickup_counts["trip_count"].mean(),
        pickup_counts["trip_count"].median(),
        pickup_counts["trip_count"].max(),
        pickup_counts["trip_count"].min()
    ]
})

display(zone_demand_summary)

zone_demand_summary.to_csv(
    TABLES_DIR / "pickup_zone_demand_summary_march_2026.csv",
    index=False
)

In [ ]:
od_counts = (
    df_mar
    .groupby(["PULocationID", "DOLocationID"])
    .size()
    .rename("trip_count")
    .reset_index()
    .rename(columns={
        "PULocationID": "PU_LocationID",
        "DOLocationID": "DO_LocationID"
    })
)

od_counts["share_percent"] = (
    od_counts["trip_count"] / len(df_mar) * 100
)

print("Number of observed OD pairs:", len(od_counts))

display(
    od_counts
    .sort_values("trip_count", ascending=False)
    .head(10)
)

In [ ]:
pickup_lookup = zones_lookup[
    ["LocationID", "Zone", "Borough"]
].rename(columns={
    "LocationID": "PU_LocationID",
    "Zone": "PU_Zone",
    "Borough": "PU_Borough"
})

dropoff_lookup = zones_lookup[
    ["LocationID", "Zone", "Borough"]
].rename(columns={
    "LocationID": "DO_LocationID",
    "Zone": "DO_Zone",
    "Borough": "DO_Borough"
})

od_named = (
    od_counts
    .merge(
        pickup_lookup,
        on="PU_LocationID",
        how="left",
        validate="many_to_one"
    )
    .merge(
        dropoff_lookup,
        on="DO_LocationID",
        how="left",
        validate="many_to_one"
    )
)

top_od_pairs = (
    od_named
    .sort_values("trip_count", ascending=False)
    .reset_index(drop=True)
)

top_od_pairs.to_csv(
    TABLES_DIR / "top_od_pairs_march_2026.csv",
    index=False
)

display(
    top_od_pairs[
        [
            "PU_Zone",
            "PU_Borough",
            "DO_Zone",
            "DO_Borough",
            "trip_count",
            "share_percent"
        ]
    ].head(20)
)

In [ ]:
top_15_od = top_od_pairs.head(15).copy()

top_15_od["route"] = (
    top_15_od["PU_Zone"].fillna("Unknown")
    + " → "
    + top_15_od["DO_Zone"].fillna("Unknown")
)

top_15_od = top_15_od.sort_values("trip_count", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))

ax.barh(
    top_15_od["route"],
    top_15_od["trip_count"]
)

ax.set_title("Top 15 Origin–Destination Pairs — March 2026")
ax.set_xlabel("Number of Trips")
ax.set_ylabel("Route")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "top_od_pairs_march_2026.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
ts = (
    daily_demand
    .sort_index()
    .astype("float64")
    .asfreq("D")
)

print("Start date:", ts.index.min())
print("End date:", ts.index.max())
print("Number of observations:", len(ts))
print("Missing observations:", ts.isna().sum())

if ts.isna().any():
    raise ValueError(
        "Daily demand contains missing dates or values. "
        "Investigate them before forecasting."
    )

display(ts.to_frame(name="trip_count").head())
display(ts.to_frame(name="trip_count").tail())

In [ ]:
test_size = 7
seasonal_period = 7

if len(ts) <= test_size + seasonal_period:
    raise ValueError(
        "Not enough observations for the selected train-test split."
    )

train = ts.iloc[:-test_size]
test = ts.iloc[-test_size:]

print("Training period:", train.index.min(), "to", train.index.max())
print("Training observations:", len(train))

print("\nTesting period:", test.index.min(), "to", test.index.max())
print("Testing observations:", len(test))

display(
    pd.DataFrame({
        "actual_trips": test
    })
)

In [ ]:
history = train.tolist()
naive_predictions = []

for actual_value in test:
    prediction = history[-1]
    naive_predictions.append(prediction)

    # After the day is observed, it becomes available for the next prediction.
    history.append(actual_value)

naive_results = pd.DataFrame(
    {
        "actual": test.to_numpy(),
        "predicted": naive_predictions
    },
    index=test.index
)

naive_results["error"] = (
    naive_results["actual"] - naive_results["predicted"]
)

naive_results["absolute_error"] = naive_results["error"].abs()

naive_results["squared_error"] = (
    naive_results["error"] ** 2
)

display(naive_results)

In [ ]:
seasonal_predictions = ts.shift(seasonal_period).loc[test.index]

seasonal_results = pd.DataFrame(
    {
        "actual": test,
        "predicted": seasonal_predictions
    },
    index=test.index
)

seasonal_results["error"] = (
    seasonal_results["actual"] - seasonal_results["predicted"]
)

seasonal_results["absolute_error"] = (
    seasonal_results["error"].abs()
)

seasonal_results["squared_error"] = (
    seasonal_results["error"] ** 2
)

display(seasonal_results)

In [ ]:
def evaluate_forecast(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    valid_mask = np.isfinite(actual) & np.isfinite(predicted)

    actual = actual[valid_mask]
    predicted = predicted[valid_mask]

    if len(actual) == 0:
        raise ValueError("No valid observations available for evaluation.")

    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))

    return {
        "MAE": mae,
        "RMSE": rmse,
        "n_observations": len(actual)
    }


naive_metrics = evaluate_forecast(
    naive_results["actual"],
    naive_results["predicted"]
)

seasonal_metrics = evaluate_forecast(
    seasonal_results["actual"],
    seasonal_results["predicted"]
)

print("Naive metrics:", naive_metrics)
print("Seasonal Naive metrics:", seasonal_metrics)

In [ ]:
forecast_comparison = pd.DataFrame(
    [
        {
            "model": "Naive",
            **naive_metrics
        },
        {
            "model": "Seasonal Naive",
            **seasonal_metrics
        }
    ]
)

forecast_comparison = forecast_comparison.sort_values(
    "MAE"
).reset_index(drop=True)

display(
    forecast_comparison.style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "n_observations": "{:.0f}"
    })
)

forecast_comparison.to_csv(
    TABLES_DIR / "forecast_model_comparison_march_2026.csv",
    index=False
)

In [ ]:
naive_mae = naive_metrics["MAE"]
seasonal_mae = seasonal_metrics["MAE"]

naive_rmse = naive_metrics["RMSE"]
seasonal_rmse = seasonal_metrics["RMSE"]

mae_improvement = (
    (seasonal_mae - naive_mae) / seasonal_mae * 100
    if seasonal_mae != 0
    else np.nan
)

rmse_improvement = (
    (seasonal_rmse - naive_rmse) / seasonal_rmse * 100
    if seasonal_rmse != 0
    else np.nan
)

print(f"Naive MAE improvement over Seasonal Naive: {mae_improvement:.2f}%")
print(f"Naive RMSE improvement over Seasonal Naive: {rmse_improvement:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(
    test.index,
    test.values,
    marker="o",
    linewidth=2,
    label="Actual Demand"
)

ax.plot(
    naive_results.index,
    naive_results["predicted"],
    marker="s",
    linestyle="--",
    label="Naive"
)

ax.plot(
    seasonal_results.index,
    seasonal_results["predicted"],
    marker="^",
    linestyle="--",
    label="Seasonal Naive"
)

ax.set_title("Actual vs Predicted Daily Taxi Demand — March 2026")
ax.set_xlabel("Date")
ax.set_ylabel("Number of Trips")
ax.legend()
ax.grid(alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "forecast_comparison_march_2026.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.axhline(0, linewidth=1)

ax.plot(
    naive_results.index,
    naive_results["error"],
    marker="o",
    label="Naive Error"
)

ax.plot(
    seasonal_results.index,
    seasonal_results["error"],
    marker="s",
    label="Seasonal Naive Error"
)

ax.set_title("Forecasting Errors — March 2026")
ax.set_xlabel("Date")
ax.set_ylabel("Actual - Predicted Trips")
ax.legend()
ax.grid(alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "forecast_errors_march_2026.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
naive_results.to_csv(
    TABLES_DIR / "naive_forecast_results_march_2026.csv",
    index_label="date"
)

seasonal_results.to_csv(
    TABLES_DIR / "seasonal_naive_forecast_results_march_2026.csv",
    index_label="date"
)

print("Forecasting outputs saved successfully.")

print("\nGenerated files:")
print(TABLES_DIR / "forecast_model_comparison_march_2026.csv")
print(TABLES_DIR / "naive_forecast_results_march_2026.csv")
print(TABLES_DIR / "seasonal_naive_forecast_results_march_2026.csv")
print(FIGURES_DIR / "forecast_comparison_march_2026.png")
print(FIGURES_DIR / "forecast_errors_march_2026.png")

In [ ]:

expected_files = {
    "tables": [
        "raw_dataset_structure.csv",
        "dataset_overview_march_2026.csv",
        "data_dictionary_march_2026.csv",
        "missing_values_march_2026.csv",
        "numeric_descriptive_statistics_march_2026.csv",
        "pearson_correlation_march_2026.csv",
        "spearman_correlation_march_2026.csv",
        "correlation_comparison_march_2026.csv",
        "daily_demand_march_2026.csv",
        "weekday_demand_march_2026.csv",
        "hourly_demand_march_2026.csv",
        "weekday_hour_demand_march_2026.csv",
        "top_pickup_zones_march_2026.csv",
        "pickup_zone_demand_summary_march_2026.csv",
        "top_od_pairs_march_2026.csv",
        "forecast_model_comparison_march_2026.csv",
        "naive_forecast_results_march_2026.csv",
        "seasonal_naive_forecast_results_march_2026.csv",
    ],
    "figures": [
        "numeric_distributions_march_2026.png",
        "passenger_count_distribution_march_2026.png",
        "multivariate_pairplot_march_2026.png",
        "correlation_heatmaps_march_2026.png",
        "daily_demand_march_2026.png",
        "weekday_demand_march_2026.png",
        "hourly_demand_march_2026.png",
        "weekday_hour_demand_march_2026.png",
        "pickup_demand_map_log_march_2026.png",
        "top_od_pairs_march_2026.png",
        "forecast_comparison_march_2026.png",
        "forecast_errors_march_2026.png",
    ],
}

output_check = []

for category, filenames in expected_files.items():
    folder = TABLES_DIR if category == "tables" else FIGURES_DIR

    for filename in filenames:
        file_path = folder / filename

        output_check.append({
            "category": category,
            "filename": filename,
            "exists": file_path.exists(),
            "size_kb": (
                round(file_path.stat().st_size / 1024, 2)
                if file_path.exists()
                else np.nan
            )
        })

output_check = pd.DataFrame(output_check)

display(output_check)

print("\nExisting files:", output_check["exists"].sum())
print("Missing files:", (~output_check["exists"]).sum())

output_check.to_csv(
    TABLES_DIR / "phase_1_output_audit.csv",
    index=False
)

In [ ]:
# Cell 68: Final validation of date range and daily demand

pickup_dates = df_mar["tpep_pickup_datetime"]

validation_results = {
    "expected_start_date": START_DATE,
    "expected_end_date_exclusive": END_DATE,
    "observed_min_pickup": pickup_dates.min(),
    "observed_max_pickup": pickup_dates.max(),
    "rows_in_march": len(df_mar),
    "rows_outside_march": int(
        (
            (pickup_dates < START_DATE)
            | (pickup_dates >= END_DATE)
        ).sum()
    ),
    "daily_series_observations": len(ts),
    "daily_series_missing_values": int(ts.isna().sum()),
    "daily_series_total_trips": int(ts.sum()),
    "filtered_dataframe_total_trips": len(df_mar),
}

validation_table = pd.DataFrame(
    list(validation_results.items()),
    columns=["check", "value"]
)

display(validation_table)

assert validation_results["rows_outside_march"] == 0, (
    "Some pickup timestamps fall outside March 2026."
)

assert validation_results["daily_series_missing_values"] == 0, (
    "The daily time series contains missing values."
)

assert validation_results["daily_series_total_trips"] == len(df_mar), (
    "Daily demand total does not match the number of filtered trips."
)

validation_table.to_csv(
    TABLES_DIR / "phase_1_validation_results.csv",
    index=False
)

print("Final validation completed successfully.")

In [ ]:
top_pickup = top_pickup_zones.iloc[0]
top_route = top_od_pairs.iloc[0]

best_model = forecast_comparison.iloc[0]

findings = [
    {
        "topic": "Dataset",
        "finding": (
            f"The March 2026 dataset contains {len(df_mar):,} trips."
        )
    },
    {
        "topic": "Data Quality",
        "finding": (
            f"{int(df_mar.duplicated().sum()):,} exact duplicate rows "
            "were identified in the filtered dataset."
        )
    },
    {
        "topic": "Trip Characteristics",
        "finding": (
            f"Median trip distance is "
            f"{df_mar['trip_distance'].median():.2f} miles."
        )
    },
    {
        "topic": "Trip Characteristics",
        "finding": (
            f"Median trip duration is "
            f"{df_mar['trip_duration_min'].median():.2f} minutes."
        )
    },
    {
        "topic": "Geography",
        "finding": (
            f"The highest-volume pickup zone is {top_pickup['Zone']} "
            f"({top_pickup['Borough']}), with "
            f"{int(top_pickup['trip_count']):,} trips."
        )
    },
    {
        "topic": "Geography",
        "finding": (
            f"The most frequent observed OD pair is "
            f"{top_route['PU_Zone']} to {top_route['DO_Zone']}, "
            f"with {int(top_route['trip_count']):,} trips."
        )
    },
    {
        "topic": "Forecasting",
        "finding": (
            f"The lowest MAE among the evaluated baseline models "
            f"was obtained by {best_model['model']}, with "
            f"MAE={best_model['MAE']:.2f} and "
            f"RMSE={best_model['RMSE']:.2f}."
        )
    },
    {
        "topic": "Limitations",
        "finding": (
            "Forecast evaluation uses only the final seven days of "
            "one month; results should be treated as preliminary."
        )
    },
]

findings_df = pd.DataFrame(findings)

display(findings_df)

findings_df.to_csv(
    TABLES_DIR / "phase_1_findings_summary.csv",
    index=False
)